In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Navigate to Google Drive
%cd /content/drive/MyDrive

# Clone repository if not already present
!git clone https://github.com/swapnasaina274/malicious-mobile-app-detection.git || true

# Enter repository folder
%cd malicious-mobile-app-detection

# Verify project files
!ls -la

Mounted at /content/drive
/content/drive/MyDrive
fatal: destination path 'malicious-mobile-app-detection' already exists and is not an empty directory.
/content/drive/MyDrive/malicious-mobile-app-detection
total 5062
drwx------ 3 root root    4096 Oct 19 16:51  .
drwx------ 2 root root    4096 Oct 19 16:58  ..
-rw------- 1 root root  238471 Oct 19 13:02  01_Machine_Learning_Model.ipynb
-rw------- 1 root root  120958 Oct 19 13:02  02_Application_Analysis.ipynb
-rw------- 1 root root  982844 Oct 19 13:02  03_Screenshot_Vision_Analysis.ipynb
-rw------- 1 root root    9203 Oct 19 13:02  04_integration_of_agentic_ai.ipynb
-rw------- 1 root root    4486 Oct 19 16:51  app.py
-rw------- 1 root root 3812499 Oct 19 13:02 'byteplot work.ipynb'
drwx------ 2 root root    4096 Oct 19 13:02  .git
-rw------- 1 root root    1072 Oct 19 13:02  README.md


In [2]:
!pip install -q langchain langchain-google-genai langchain-community pydantic python-dotenv

In [8]:
import os

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6IQNstUPMvHzTjnUU_ciuzQhseLw9TziCNlmPspt8l2rA"

In [9]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from IPython.display import display, Markdown

# 1. Define Static Analysis Permission Tool
@tool
def analyze_permission_risk(permissions: list) -> str:
    """Evaluates Android permissions for dangerous cross-permission combinations."""
    dangerous_combos = {
        ("SEND_SMS", "READ_CONTACTS"): "CRITICAL: Potential SMS Botnet or Contact Exfiltration.",
        ("SYSTEM_ALERT_WINDOW", "BIND_ACCESSIBILITY_SERVICE"): "CRITICAL: Potential Banking Overlay / Phishing Attack.",
        ("RECORD_AUDIO", "INTERNET"): "HIGH: Potential Spyware / Eavesdropping.",
        ("READ_EXTERNAL_STORAGE", "INTERNET"): "MEDIUM: Potential Data Exfiltration."
    }

    findings = []
    normalized_perms = [p.strip().upper() for p in permissions]
    for combo, risk_desc in dangerous_combos.items():
        if all(p in normalized_perms for p in combo):
            findings.append(f"- Combo {combo}: {risk_desc}")

    return "\n".join(findings) if findings else "No high-risk permission combinations detected."

# 2. Setup Gemini Model
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

# 3. Inputs from Phase 1 & 2 Models
ml_predicted_class = "SMS Malware"
ml_confidence = 0.9412
extracted_permissions = ["SEND_SMS", "READ_CONTACTS", "INTERNET"]

# 4. Agent Tool Execution & Prompt Synthesis
permission_analysis = analyze_permission_risk.invoke({"permissions": extracted_permissions})

prompt = f"""
You are an expert Mobile Security Analysis AI Agent.
Evaluate the app data provided by Phase 1 and Phase 2 ML/DL models:

App Security Input Data:
- ML Prediction Class: {ml_predicted_class} (Confidence: {ml_confidence * 100:.2f}%)
- Extracted Permissions: {extracted_permissions}
- Tool Inspection Findings: {permission_analysis}

Generate a comprehensive Security Risk Assessment report containing:
1. Overall Risk Severity Level (Low, Medium, High, Critical)
2. Detailed Permission & Behavioral Risk Explanation
3. Clear Actionable Recommendations for the end user
"""

# 5. Generate Agent Assessment Report
response = llm.invoke(prompt)

# Extract and display clean Markdown text
if isinstance(response.content, list):
    report_text = "".join([block.get("text", "") for block in response.content if isinstance(block, dict)])
else:
    report_text = str(response.content)

display(Markdown(report_text))

# Mobile Security Risk Assessment Report

---

## 1. Overall Risk Severity Level
### **CRITICAL**

**Summary Justification:**  
The Machine Learning/Deep Learning engine has classified this application as **SMS Malware** with a high confidence score of **94.12%**. The requested permissions (`SEND_SMS`, `READ_CONTACTS`, and `INTERNET`) form a classic, high-risk triage commonly leveraged by mobile botnets, spyware, and SMS toll-fraud trojans.

---

## 2. Detailed Permission & Behavioral Risk Explanation

### Dangerous Permission Breakdown
* **`READ_CONTACTS`**: Grants the application full access to the stored address book, including contact names, phone numbers, email addresses, and communication history.
* **`SEND_SMS`**: Allows the application to transmit SMS messages autonomously in the background without explicit real-time user interaction or visible confirmation.
* **`INTERNET`**: Provides the application with unrestricted access to transmit and receive network traffic over Wi-Fi and mobile data.

### Threat Vectors & Behavioral Analysis

1. **Worm-Like Propagation (Smishing Botnet):**
   * By combining `READ_CONTACTS` with `SEND_SMS`, the malware can query the user's contacts and silently send deceptive SMS messages containing malicious download links or phishing lures to everyone in the address book. Because these texts originate from a trusted number, infection rates among secondary victims are significantly higher.
2. **Contact Exfiltration & Espionage:**
   * The pairing of `READ_CONTACTS` and `INTERNET` allows the application to harvest private contact databases and upload them directly to a remote Command and Control (C2) server for monetization, social engineering campaigns, or identity theft.
3. **SMS Toll Fraud / Financial Drain:**
   * Attackers frequently use background `SEND_SMS` capabilities to subscribe users to premium-rate messaging services without their consent, leading to unexpected and costly carrier charges.
4. **C2 Coordination & Telemetry:**
   * The `INTERNET` permission enables the application to receive dynamic commands from threat actors (e.g., target lists, message templates, activation triggers), allowing it to evade basic signature-based detection.

---

## 3. Clear Actionable Recommendations for the End User

### Immediate Containment Steps
1. **Uninstall the Application Immediately:**
   * Go to **Settings > Apps > [App Name] > Uninstall**. 
   * If the app has acquired Device Administrator privileges and prevents standard uninstallation, boot the device into **Safe Mode** to revoke administrator access and remove the app.
2. **Disconnect Network Access:**
   * Place the phone in **Airplane Mode** immediately if active background activity or unexplained SMS transmissions are suspected.

### Mitigation & Verification Steps
3. **Monitor Mobile Carrier Invoices:**
   * Log into your mobile carrier account and review recent SMS logs and billing statements for unauthorized premium SMS charges or bulk outbound text surges. Contact your carrier immediately to dispute fraudulent charges and request a block on premium-rate numbers.
4. **Notify Contacts (If Messages Were Sent):**
   * If unauthorized SMS messages were sent from your device, alert your contacts not to click any links or download attachments sent from your number.
5. **Execute a Full Antivirus Scan:**
   * Run a deep system scan using a reputable Google Play Protect-certified mobile security solution (e.g., Bitdefender, Sophos, Malwarebytes).
6. **Consider a Factory Reset (Worst-Case Precaution):**
   * If the app re-installs itself or exhibits persistent root-level anomalies, back up essential personal files (excluding APKs) and perform a full **Factory Data Reset**.